# Turkish Food Name GeneratorTwo deliverables, built on the geographical indications registered at TÜRKPATENT:1. **A BPE tokenizer**, trained from scratch on our own text (`model/bpe_tokenizer.py`)2. **A miniature Qwen3-style transformer** that generates new dish names (`model/`)The data pipeline that feeds this notebook:| File | What it is || --- | --- || `data/foods_raw.txt` | 500 registered food names, as scraped || `data/clean_foods.py` | lowercasing, province stripping, dedupe || `data/foods_clean.txt` | 484 clean names, one per line |

In [ ]:
import math
import sys

sys.path.insert(0, "model")

import torch

from bpe_tokenizer import BPETokenizer
from config import ModelConfig
from model import TinyQwen
from tokenizer import CharTokenizer

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. The dataRegistered names are things like *"Diyarbakır Patlıcanlı Kazan Kebabı"*. `clean_foods.py`lowercases them with the Turkish `I/İ` rule and drops the province prefix, but only whentwo or more words survive: in *"Adana Kebabı"* the possessive `-ı` binds to *Adana*, sostripping it would leave the ungrammatical *"kebabı"*.

In [ ]:
DATA = "data/foods_clean.txt"

text = open(DATA, encoding="utf-8").read()
names = [n for n in text.splitlines() if n]

print(f"{len(names)} names, {len(text)} characters")
print(f"alphabet ({len(set(text))}): {''.join(sorted(set(text)))!r}")
names[:8]

## 2. The BPE tokenizerByte-pair encoding starts from single characters and repeatedly merges the most frequentadjacent pair. Two rules shape ours:- **Merges never cross whitespace.** Words are merged in isolation, so no single token can  swallow a whole dish name and let the model memorise it.- **`\n` stays a standalone token.** It marks the start and end of a name, so it is what  the model uses to learn when to stop.`vocab_size=200` is not arbitrary — see the sweep two cells down.

In [ ]:
tok = BPETokenizer.from_file(DATA)

print(f"vocab_size = {tok.vocab_size}  ({len(tok.chars)} chars + {len(tok.merges)} merges)")
print(f"eos_id     = {tok.eos_id}  ({tok.itos[tok.eos_id]!r})")
print()
print("merges it learned, in order:")
print("  first 12:", [''.join(p) for p in tok.merges[:12]])
print("  last  12:", [''.join(p) for p in tok.merges[-12:]])

In [ ]:
for name in ["adana kebabı", "içli köftesi", "patlıcanlı kazan kebabı", "ekşili çorbası"]:
    print(f"{name:26} -> {[tok.itos[i] for i in tok.encode(name)]}")

assert tok.decode(tok.encode(text)) == text, "roundtrip must be exact"
print("\nroundtrip over the whole corpus: exact")

### Why 200?The corpus is only ~7.4k characters. A bigger vocabulary buys shorter sequences but starveseach token of examples — and an embedding seen three times is memorised, not learned. Thecliff sits between 200 and 300.

In [ ]:
from collections import Counter

print(f"{'vocab':>6} {'tokens/name':>12} {'avg seen':>9} {'seen <5x':>9}")
for size in [30, 100, 200, 300, 400, 590]:
    t = BPETokenizer.from_file(DATA, vocab_size=size)
    ids = t.encode(text)
    freq = Counter(ids)
    rare = sum(1 for i in range(t.vocab_size) if freq[i] < 5)
    per_name = sum(len(t.encode(n)) for n in names) / len(names)
    print(f"{t.vocab_size:>6} {per_name:>12.1f} {len(ids) / t.vocab_size:>9.1f} "
          f"{rare * 100 // t.vocab_size:>8}%")

### BPE vs character-levelThe repository this project builds on is character-level. Same data, same task — this iswhat BPE buys us.

In [ ]:
char_tok = CharTokenizer.from_file(DATA)

char_len = sum(len(char_tok.encode(n)) for n in names) / len(names)
bpe_len = sum(len(tok.encode(n)) for n in names) / len(names)

print(f"char-level: vocab {char_tok.vocab_size:3}   {char_len:.1f} tokens/name")
print(f"BPE       : vocab {tok.vocab_size:3}   {bpe_len:.1f} tokens/name   "
      f"({char_len / bpe_len:.1f}x shorter)")

## 3. The model`TinyQwen` is the dense Qwen3 recipe in miniature: RMSNorm, QK-Norm, grouped-queryattention, RoPE, SwiGLU. `vocab_size` flows in from the tokenizer.Going from a 30-character vocabulary to 200 BPE tokens exposed one bug worth fixing.`nn.Embedding` defaults to `N(0, 1)`, and weight tying makes that matrix the output head aswell, so logits started with std ~`sqrt(hidden_size)` and step 1 sat at loss **32** insteadof the uniform baseline of 5.30. `model.py` now initialises weights at `std=0.02` (the GPT-2value), which halves the steps needed and ends ~20% lower:| step | default `N(0, 1)` | `std=0.02` || --- | --- | --- || 1 | 31.89 | 5.30 || 1000 | 1.24 | 0.62 || 3000 | 0.47 | 0.38 |

In [ ]:
# The longest name is 18 tokens; +1 for the leading newline. A shorter window would
# never show the model a complete name.
BLOCK_SIZE = 20

lengths = [len(tok.encode(n)) for n in names]
print(f"longest name: {max(lengths)} tokens -> BLOCK_SIZE={BLOCK_SIZE}, "
      f"max_seq_len={ModelConfig().max_seq_len}")

data = torch.tensor(tok.encode(text), dtype=torch.long)
cfg = ModelConfig(vocab_size=tok.vocab_size)
model = TinyQwen(cfg).to(device)

print(f"parameters  : {sum(p.numel() for p in model.parameters()):,}")
cfg

In [ ]:
BATCH_SIZE = 64
STEPS = 3000
LEARNING_RATE = 3e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def get_batch():
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + 1 + BLOCK_SIZE] for i in ix])
    return x.to(device), y.to(device)


for step in range(1, STEPS + 1):
    x, y = get_batch()
    _, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0 or step == 1:
        print(f"step {step:5d}  loss {loss.item():.4f}")

print(f"\nuniform baseline (ln {tok.vocab_size}): {math.log(tok.vocab_size):.4f}")

## 4. Generating new namesEvery name starts from the newline token and is sampled one token at a time until the modelemits newline again.The interesting part is not whether the output is Turkish — it is whether the name is**new**. `adana kebabı` coming back out is memorisation, not generation, so every sample ischecked against the training set.

In [ ]:
known = set(names)


@torch.no_grad()
def generate(n: int = 20, temperature: float = 0.9, top_k: int = None) -> list[str]:
    model.eval()
    start = torch.full((n, 1), tok.eos_id, dtype=torch.long, device=device)
    out = model.generate(start, max_new_tokens=cfg.max_seq_len,
                         temperature=temperature, top_k=top_k, eos_id=tok.eos_id)
    model.train()
    return [tok.decode(row[1:]).split("\n")[0].strip() for row in out.tolist()]


for name in generate(20):
    if name:
        print(f"{'copy' if name in known else 'NEW ':4}  {name}")

In [ ]:
samples = [n for n in generate(300) if n]
novel = [n for n in samples if n not in known]

print(f"novel: {len(novel)}/{len(samples)} ({len(novel) * 100 // len(samples)}%)")
print()
for name in novel[:15]:
    print("  ", name)

Temperature controls the trade-off: low values replay the training set, high values inventwords that stop being Turkish.

In [ ]:
for temperature in [0.6, 0.8, 1.0, 1.2]:
    samples = [n for n in generate(120, temperature=temperature) if n]
    novel = sum(1 for n in samples if n not in known)
    print(f"T={temperature}  novel {novel * 100 // len(samples):>3}%   "
          f"e.g. {', '.join(samples[:3])}")

## 5. Save

In [ ]:
torch.save({"model": model.state_dict(), "chars": tok.chars,
            "merges": tok.merges, "cfg": cfg}, "model/tiny_food.pt")
tok.save("model/merges.json")

print("saved model/tiny_food.pt and model/merges.json")